# Fink Alert Stream with Kafka

This notebook demonstrates how to receive alerts from Fink using Kafka and save transient information and classification to JSON files.

## About Fink

[Fink](https://fink-broker.org/) is a broker infrastructure enabling a wide range of applications and services to connect to large streams of alerts issued from telescopes all over the world. Fink processes and redistributes alerts from the [Zwicky Transient Facility (ZTF)](https://www.ztf.caltech.edu/) and will soon process alerts from the [Vera C. Rubin Observatory](https://rubinobservatory.org/).

### Key Features:
- Real-time classification of transient events
- Multi-messenger astronomy support
- Access to ZTF alerts with added value from Fink processing
- Kafka-based streaming for efficient data distribution


### Let's first have a walk through the [doc page](https://fink-broker.readthedocs.io/en/latest/)

## Prerequisites

**Important**: This notebook requires credentials to access Fink streams. You need to:

1. Subscribe to Fink streams by filling [this form](https://forms.gle/2td4jysT4e9pkf889)
2. Register your credentials using:
   ```bash
   fink_client_register -username <USERNAME> -group_id <GROUP_ID> -servers <SERVERS>
   ```

**Note**: This notebook must be run locally or on Colab (not on Binder) as it requires secure connections.

## Installation

To run locally:
```bash
pip install fink-client
```

To run on Colab, uncomment and run the following cell:

In [ ]:
# ! pip install fink-client

## Import Required Libraries

In [ ]:
import os
import json
import datetime
import pandas as pd
from fink_client.consumer import AlertConsumer
from fink_client.configuration import load_credentials

## Configure the Fink Consumer

The consumer will use the credentials stored in `~/.finkclient/credentials.yml`

In [ ]:
# Load credentials from the configuration file
# If you haven't registered yet, run: fink_client_register -h for help

# Available topics (examples):
# - fink_early_sn_candidates_ztf: Early supernova candidates
# - fink_sso_ztf_candidates_ztf: Solar System Object candidates
# - fink_kn_candidates_ztf: Kilonova candidates
# - fink_early_sn_ia_candidates_ztf: Type Ia supernova candidates

# You can check available topics by running in terminal:
# fink_consumer --available_topics

topics = ['fink_early_sn_candidates_ztf']  # Change this to your topic of interest

## Initialize the Alert Consumer

In [ ]:
# Create AlertConsumer instance
# The configuration is automatically loaded from ~/.finkclient/credentials.yml

consumer_config = {
    'group.id': 'your_group_id',  # Replace with your group_id from credentials
    'auto.offset.reset': 'earliest'  # 'earliest' to get all messages, 'latest' for new ones only
}

# Uncomment and configure when you have credentials:
# mapcache = load_credentials()
# myconfig = consumer_config
# consumer = AlertConsumer(topics, myconfig, mapcache=mapcache)

## Create Output Directory for Alerts

In [ ]:
output_dir = "./fink_alerts"
os.makedirs(output_dir, exist_ok=True)
print(f"Alerts will be saved to: {output_dir}")

## Function to Extract and Save Alert Information

In [ ]:
def process_fink_alert(alert_data, output_dir="./fink_alerts"):
    """
    Extract key information and transient classification from a Fink alert
    and save to JSON file.
    
    Parameters
    ----------
    alert_data : dict
        Alert data from Fink stream
    output_dir : str
        Directory to save JSON files
    
    Returns
    -------
    dict
        Processed alert information
    """
    try:
        # Extract basic information
        alert_info = {
            # Object identification
            'objectId': alert_data.get('objectId'),
            'candid': alert_data.get('candid'),
            
            # Position
            'ra': alert_data.get('candidate', {}).get('ra'),
            'dec': alert_data.get('candidate', {}).get('dec'),
            
            # Time
            'jd': alert_data.get('candidate', {}).get('jd'),
            'timestamp': alert_data.get('timestamp'),
            
            # Brightness and photometry
            'magpsf': alert_data.get('candidate', {}).get('magpsf'),
            'sigmapsf': alert_data.get('candidate', {}).get('sigmapsf'),
            'fid': alert_data.get('candidate', {}).get('fid'),  # Filter ID
            
            # Fink classifications and added value
            'cdsxmatch': alert_data.get('cdsxmatch'),  # Cross-match with catalogs
            'roid': alert_data.get('roid'),  # Solar system object classification
            'mulens': alert_data.get('mulens'),  # Microlensing classification
            'snn_snia_vs_nonia': alert_data.get('snn_snia_vs_nonia'),  # SN Ia vs non-Ia
            'snn_sn_vs_all': alert_data.get('snn_sn_vs_all'),  # SN vs all
            'rf_snia_vs_nonia': alert_data.get('rf_snia_vs_nonia'),  # Random Forest classifier
            'rf_kn_vs_nonkn': alert_data.get('rf_kn_vs_nonkn'),  # Kilonova classifier
            
            # Additional metadata
            'classtar': alert_data.get('candidate', {}).get('classtar'),  # Star/galaxy classifier
            'drb': alert_data.get('candidate', {}).get('drb'),  # Deep learning RB score
            'ndethist': alert_data.get('candidate', {}).get('ndethist'),  # Number of detections
        }
        
        # Create output directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)
        
        # Generate filename
        object_id = alert_info.get('objectId', 'unknown')
        candid = alert_info.get('candid', datetime.datetime.now().strftime("%Y%m%d_%H%M%S"))
        filename = f"fink_{object_id}_{candid}.json"
        filepath = os.path.join(output_dir, filename)
        
        # Save to JSON file
        with open(filepath, 'w') as f:
            json.dump(alert_info, f, indent=2)
        
        print(f"Saved alert to: {filepath}")
        print(f"  Object: {alert_info['objectId']}")
        print(f"  Position: RA={alert_info['ra']:.6f}, Dec={alert_info['dec']:.6f}")
        print(f"  Classification: {alert_info.get('cdsxmatch', 'Unknown')}")
        print("-" * 50)
        
        return alert_info
        
    except Exception as e:
        print(f"Error processing Fink alert: {e}")
        return None

## Poll Alerts from Fink Stream

This section demonstrates how to poll alerts from the Fink stream. You can set a limit to download a specific number of alerts.

In [ ]:
# Set the number of alerts to download (None for all available)
max_alerts = 10

# Uncomment to run when you have credentials:
# alert_count = 0
# 
# try:
#     while True:
#         topic, alert, key = consumer.poll(timeout=10)
#         
#         if topic is None:
#             print("No more alerts available")
#             break
#         
#         # Process and save the alert
#         process_fink_alert(alert, output_dir=output_dir)
#         
#         alert_count += 1
#         
#         if max_alerts and alert_count >= max_alerts:
#             print(f"Reached maximum number of alerts: {max_alerts}")
#             break
#             
# except KeyboardInterrupt:
#     print("Interrupted by user")
# finally:
#     consumer.close()
#     print(f"\nTotal alerts processed: {alert_count}")

## Analyze Saved Alerts

Load and analyze the saved alerts from JSON files.

In [ ]:
def load_alerts_from_directory(directory="./fink_alerts"):
    """
    Load all alert JSON files from a directory into a pandas DataFrame.
    
    Parameters
    ----------
    directory : str
        Directory containing alert JSON files
    
    Returns
    -------
    pd.DataFrame
        DataFrame containing all alerts
    """
    alerts = []
    
    if not os.path.exists(directory):
        print(f"Directory {directory} does not exist")
        return pd.DataFrame()
    
    for filename in os.listdir(directory):
        if filename.endswith('.json'):
            filepath = os.path.join(directory, filename)
            with open(filepath, 'r') as f:
                alert_data = json.load(f)
                alerts.append(alert_data)
    
    if alerts:
        df = pd.DataFrame(alerts)
        print(f"Loaded {len(df)} alerts")
        return df
    else:
        print("No alerts found")
        return pd.DataFrame()

# Load the alerts
# alerts_df = load_alerts_from_directory(output_dir)
# alerts_df.head()

## Summary Statistics of Classifications

In [ ]:
# Uncomment to analyze when you have alerts:
# if not alerts_df.empty:
#     print("Classification Distribution:")
#     print(alerts_df['cdsxmatch'].value_counts())
#     print("\nBasic Statistics:")
#     print(f"Total unique objects: {alerts_df['objectId'].nunique()}")
#     print(f"Magnitude range: {alerts_df['magpsf'].min():.2f} - {alerts_df['magpsf'].max():.2f}")
#     print(f"Time range: {alerts_df['jd'].min():.2f} - {alerts_df['jd'].max():.2f}")

## Real-time Listener (Optional)

Set up a background listener that continuously monitors the Fink stream.

In [ ]:
import threading
import time

def fink_listener(consumer, output_dir="./fink_alerts", stop_event=None):
    """
    Background listener function for continuous monitoring.
    
    Parameters
    ----------
    consumer : AlertConsumer
        Fink consumer instance
    output_dir : str
        Directory to save alerts
    stop_event : threading.Event
        Event to signal when to stop listening
    """
    try:
        while not stop_event.is_set():
            topic, alert, key = consumer.poll(timeout=10)
            
            if topic is not None:
                process_fink_alert(alert, output_dir=output_dir)
            
            time.sleep(0.1)  # Small delay to prevent CPU overuse
            
    except Exception as e:
        print(f"Error in listener: {e}")
    finally:
        consumer.close()
        print("Listener stopped")

# Uncomment to start background listener:
# stop_event = threading.Event()
# listener_thread = threading.Thread(
#     target=fink_listener, 
#     args=(consumer, output_dir, stop_event),
#     daemon=True
# )
# listener_thread.start()
# print("Background listener started")

## Check if Listener is Active

In [ ]:
# Uncomment to check:
# listener_thread.is_alive()

## Stop the Listener

In [ ]:
# Uncomment to stop:
# stop_event.set()
# listener_thread.join()
# print("Listener stopped successfully")

## Additional Resources

- [Fink Portal](https://fink-portal.org/)
- [Fink Documentation](https://fink-broker.readthedocs.io/)
- [Fink Client GitHub](https://github.com/astrolabsoftware/fink-client)
- [ZTF Science Data System](https://www.ztf.caltech.edu/)